
# Notebook 2 — The Discount Factor & Tipping Points

**Course notebook** using `DICE.py`.  
We work on two blocks from the slides: **“The discount factor”** and **“Tipping points.”**

## Summary of the notebook
- **0)** Load `DICE.py` and run a baseline.
- **1)** **The discount factor controversy**  
  1-A) From discount factor to interest rate (and quick coding exercise).  
  1-B) Real-world interest rates; pick 3 values for `p.rho` (including Stern’s low value).  
  1-C) Compute the optimal policy for the 3 scenarios.  
  1-D) Plot and compare **abatement**, **carbon tax**, **damages**, **temperatures**.  
  1-E) Short comments.
- **2)** **The role of tipping points**  
  2-A) Using `p.user_damage_fn` to define custom damages (baseline Nordhaus; double coefficient).  
  2-B) Plot outcomes.  
  2-C) Interpret.  
  2-D) Implement a **Weitzman** damage function and solve.  
  2-E) Compare figures.  
  2-F) Comment.  
  2-G) Add a **kink** at 3°C (damages double above the threshold) and interpret.



## 0) Load `DICE.py`
Make sure `DICE.py` sits next to this notebook (or adjust the import path accordingly).


In [ ]:
# Run this cell once to check/install the Python packages required for this notebook.

import sys
import subprocess
import importlib.util

required = {
    "matplotlib": "matplotlib",
    "numba": "numba",
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "tqdm": "tqdm",
}

missing = [
    package
    for module, package in required.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print("Installing:", ", ".join(missing))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", *missing]
    )

print("Python environment ready.")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from DICE import (
    Params,
    init_states,
    update_path,
    mat_to_df,
    obj_fun,
    run_optimal_policy,
)

# Baseline initialization
p       = Params()
sim     = init_states(p)
timevec = range(1, p.nT)
sim     = update_path(sim, timevec, p)

# Peek at the dataframe if you want
df = mat_to_df(sim, p)
df.head()



## 1) The Discount Factor Controversy

**Context.**  
The discount factor determines how we value the future. A higher pure rate of time preference $\rho$ means we discount the future more heavily—typically implying **lower** near-term abatement. A lower $\rho$ (as advocated by **Stern**) implies **strong** and **early** mitigation.

- **Nordhaus** (e.g., DICE): often uses $\rho \approx 1.5\%$ per year (plus growth & curvature terms in the consumption Euler equation).  
- **Stern (2006, Stern Review)**: argues for a **near-zero** pure rate of time preference ($\rho \approx 0.1\%$ per year).  
  Link to the **Stern Review**: https://webarchive.nationalarchives.gov.uk/ukgwa/20100407172811/http://www.hm-treasury.gov.uk/stern_review_report.htm

This debate is ethical as much as technical: how much should current generations **value future generations' welfare**?



### 1-A) From discount factor to interest rate

Let $\beta = \dfrac{1}{1+\rho}$ denote the planner’s discount factor. Under standard assumptions, the (approximate) real interest rate is
$$
r \approx \rho + \gamma g,
$$
where $g$ is per-capita consumption growth and $\gamma$ is the CRRA parameter (inverse of IES).

**Exercise.**
1. Read the current discounting parameters from `p`: `p.rho` and `p.gamma`.
2. Assume a plausible growth rate of consumption, e.g. $g=2\%$.
3. Compute and print $\beta$ and $r$. (Feel free to experiment with other values of $g$.)


In [ ]:
rho = p.rho
gamma = p.gamma
g = 0.02

# Compute both requested quantities:
# beta = 1 / (1 + rho)
# r = rho + gamma * g



### 1-B) Real-world interest rates & choosing three $\rho$ values

Long-run **real** interest rates are typically in the ballpark of 1–4% annually (varying by country and period). One can have a glance at:  
- US data on long term government bonds https://fred.stlouisfed.org/series/IRLTLT01USA156N
- Or a global R https://cepr.org/voxeu/columns/global-r

For this exercise, consider three values for the pure rate of time preference:
- **Nordhaus-like**: $\rho = 0.015$ (1.5%)
- **Intermediate**: $\rho = 0.005$ (0.5%)
- **Stern-like**: $\rho = 0.001$ (0.1%) — often summarized as $\beta \approx 0.999$.

We will solve the optimal policy for each. Define here simply the corresponding $\rho$.

First, change the calibration of rho for the other sceanrios.


In [ ]:
# >>> Your code here <<<
pNordhaus     = Params()
pIntermediate = Params()
pStern        = Params()
# a twist to get calculation manageable, otherwise the planner windows expands a lot
pIntermediate.toly   = 0.01
pStern.toly          = 0.01
# ...


### 1-C) Compute the optimal policy for the three scenarios

We optimize over **one control** to reduce the computation time: the abatement rate $\mu_t$.


In [ ]:
# >>> Your code here <<<
bounds_s   = [(0, 1)]
control_id = [pNordhaus.i_mu]
path_opt_Nordhaus = run_optimal_policy(sim.copy(), timevec, pNordhaus, bounds_s, control_id)
# ...


### 1-D) Plot: abatement, carbon tax, damages, temperatures

**Exercise.** Create comparison plots across the three discounting scenarios.


In [ ]:
# >>> Your code here <<<

plt.subplot(1, 4, 1)
# plot abatement
# var name: "mu"

plt.subplot(1, 4, 2)
# plot damages
# damages = p.a2 * (sim[0:,p.i_T_AT] ** p.a3)

plt.subplot(1, 4, 3)
# plot carbon tax
# var name: "Tax"

plt.subplot(1, 4, 4)
# plot temperatures
# var name: "T_AT"

plt.show()



### 1-E) Comment (short paragraph)

**Exercise.** Write a short paragraph:
- How does lowering $\rho$ alter abatement paths and taxes?  
- What happens to temperatures and damages?  
- Do your results align with the intuition from the slides?


> ✍️ You written answer here.


## 2) The Role of Tipping Points

**Slides recap.**  
Tipping points refer to abrupt, potentially irreversible shifts in the climate system (e.g., ice-sheet collapse, AMOC changes, permafrost thaw). They imply **nonlinear** and **fat-tailed** risks. Smooth quadratic damages may understate extreme outcomes.



### 2-A) Double the baseline quadratic damages

The DICE module represents damages through parameters in `Params`. The
baseline specification is

$$D(T)=a_2T^{a_3}.$$

Create `pB = Params()` for the baseline and `pN = Params()` for the alternative,
then set `pN.a2 = 2 * pN.a2`. Initialise and optimise both paths over the
abatement control `i_mu` using the same horizon and bounds.


In [ ]:
pB = Params()
pN = Params()
pN.a2 = 2 * pN.a2

timevec = range(1, p.nT)
simB = init_states(pB)
simN = init_states(pN)
bounds_mu = (0, 1)
control_id = [pB.i_mu]

# Optimise simB with pB and simN with pN using run_optimal_policy.



### 2-B) Plot outcomes (damages, tax, abatement)

**Exercise.** Compare **baseline** vs **doubled** damages on damages, tax, and abatement (assuming implementation of optimal abatement).


In [ ]:
# Compare simB (baseline) with simN (doubled a2) in four panels.
# Use columns i_mu, i_Tax and i_T_AT.
# Compute damage shares as p.a2 * temperature ** p.a3 for each calibration.



### 2-C) Interpret (short paragraph)

**Exercise.** Explain how higher damages affect optimal policy:
- How do tax and abatement shift over time?
- How does the temperature trajectory react?


> ✍️ You written answer here.

### 2-D) Weitzman-style damages (fat tails)

Use the second damage term already implemented in `DICE.py`:

$$D_W(T)=a_2T^{a_3}+a_4T^{a_5}.$$

For the Weitzman calibration, set:

```python
pW.a4 = 5.0703e-06
pW.a5 = 6.754
pW.a6 = 0.0
```

Here `a6=0` activates the additional term for positive temperatures. Simulate
the optimal path and, for comparison, a laissez-faire path without optimisation.


In [ ]:
pW = Params()
pW.a4 = 5.0703e-06
pW.a5 = 6.754
pW.a6 = 0.0

# Initialise an optimal Weitzman path and a laissez-faire copy.
# Optimise the first with run_optimal_policy and propagate the second with update_path.


### 2-E) Compare Nordhaus and Weitzman outcomes

Compare `simB`, `simW` and the laissez-faire `simW_LF`. Plot abatement,
damage shares, the carbon tax and atmospheric temperature. For Weitzman damages,
include both `a2*T**a3` and `a4*T**a5`.


In [ ]:
# Use simB, simW and simW_LF from the previous cells.
# Build four panels for i_mu, damages, i_Tax and i_T_AT.



### 2-F) Comment (short paragraph)

**Exercise.** Discuss the policy differences implied by fatter tails:
- Are taxes/abatement higher and earlier?
- How sensitive are results to \(\kappa\)? Try a few values.


> ✍️ You written answer here.

### 2-G) Introduce a kink at 3°C

The second DICE damage term is threshold-activated. To double quadratic
damages only above 3°C, set:

```python
pG.a4 = pG.a2
pG.a5 = pG.a3
pG.a6 = 3.0
```

Optimise this calibration, build its laissez-faire counterpart, compare both
with the baseline, and interpret the response of abatement and the carbon tax.


In [ ]:
pG = Params()
pG.a4 = pG.a2
pG.a5 = pG.a3
pG.a6 = 3.0

# Initialise simG and simG_LF, optimise simG and propagate simG_LF.
# Then compare them with simB.


> ✍️ You written answer here.